# 11 GHSL Reusable Urban Growth Model

## Phase 1: Six-city inventory and modeling preflight

This first notebook phase is deliberately read-only. It verifies the exact HDFS city slugs, all 60 enriched epoch partitions, 4,900 rows per partition, unique grid keys, and schema compatibility before any longitudinal dataset is written.

**Training cities:** Bengaluru, Hyderabad, Pune Metropolitan Region, Chennai Metropolitan Region, Mumbai Metropolitan Region, and Delhi NCR.  
**Held-out city:** Jaipur. Jaipur is not used in this notebook.

The notebook uses `/content/drive/MyDrive/remote_exec.sh` as the single source of truth for the EC2 host, user, SSH key, and SSH options. Google Drive is mounted because it also stores the persistent inventory and modeling artifacts.

Run one cell at a time. Do not skip a failed gate.

In [1]:
# ============================================================
# CELL 1: MOUNT GOOGLE DRIVE AND LOCAL CONFIGURATION
# ============================================================

from google.colab import drive
from pathlib import Path
import base64
import json
import os
import re
import subprocess
import time

# Google Drive is still required because remote_exec.sh is the
# single source of truth for the current EC2 connection details.
drive.mount(
    "/content/drive",
    force_remount=False,
)

REMOTE_EXEC = Path(
    "/content/drive/MyDrive/remote_exec.sh"
)

LOCAL_ARTIFACT_ROOT = Path(
    "/content/drive/MyDrive/GHSL_Urban_Growth_Model"
)

if not REMOTE_EXEC.is_file():
    raise FileNotFoundError(
        "remote_exec.sh was not found at: "
        + str(REMOTE_EXEC)
    )

REMOTE_EXEC.chmod(0o700)

LOCAL_ARTIFACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EXPECTED_EPOCHS = [
    1975,
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
    2020,
]

EXPECTED_ROWS_PER_PARTITION = 4900
EXPECTED_CITY_COUNT = 6
EXPECTED_PARTITION_COUNT = 60
EXPECTED_TOTAL_ROWS = 294000

TRAINING_CITY_HINTS = {
    "Bengaluru": [
        "bengaluru",
        "bangalore",
    ],
    "Hyderabad": [
        "hyderabad",
    ],
    "Pune Metropolitan Region": [
        "pune",
    ],
    "Chennai Metropolitan Region": [
        "chennai",
    ],
    "Mumbai Metropolitan Region": [
        "mumbai",
    ],
    "Delhi NCR": [
        "delhi",
        "ncr",
    ],
}

print("Google Drive mounted: PASS")
print("Remote executor:", REMOTE_EXEC)
print("Persistent artifact root:", LOCAL_ARTIFACT_ROOT)
print("Expected cities: 6")
print("Expected enriched partitions: 60")
print("Expected grid-year rows: 294000")
print("Jaipur included in training: NO")
print("HDFS mutation enabled: NO")
print("CELL 1: SUCCESS")

Mounted at /content/drive
Google Drive mounted: PASS
Remote executor: /content/drive/MyDrive/remote_exec.sh
Persistent artifact root: /content/drive/MyDrive/GHSL_Urban_Growth_Model
Expected cities: 6
Expected enriched partitions: 60
Expected grid-year rows: 294000
Jaipur included in training: NO
HDFS mutation enabled: NO
CELL 1: SUCCESS


In [2]:
# ============================================================
# PREPARE TEMPORARY COLAB SSH KEY FOR remote_exec.sh
# ============================================================

import shutil


DRIVE_SSH_KEY = (
    Path("/content/drive/MyDrive")
    / "hadoop-hdfs.pem"
)

COLAB_SSH_KEY = Path(
    "/content/hadoop-hdfs.pem"
)


if not DRIVE_SSH_KEY.is_file():
    raise FileNotFoundError(
        "The persistent SSH key was not found at: "
        + str(DRIVE_SSH_KEY)
    )


# Always refresh the runtime copy. The /content filesystem is
# temporary and is reset whenever the Colab runtime restarts.

if COLAB_SSH_KEY.exists():
    COLAB_SSH_KEY.unlink()


shutil.copy2(
    DRIVE_SSH_KEY,
    COLAB_SSH_KEY,
)


COLAB_SSH_KEY.chmod(
    0o600
)


if not COLAB_SSH_KEY.is_file():
    raise FileNotFoundError(
        "The SSH key was not copied to: "
        + str(COLAB_SSH_KEY)
    )


if COLAB_SSH_KEY.stat().st_size <= 0:
    raise RuntimeError(
        "The copied SSH key is empty: "
        + str(COLAB_SSH_KEY)
    )


print(
    "Drive SSH key:",
    DRIVE_SSH_KEY,
)

print(
    "Colab SSH key:",
    COLAB_SSH_KEY,
)

print(
    "Colab SSH key bytes:",
    COLAB_SSH_KEY.stat().st_size,
)

print(
    "Colab SSH key preparation: PASS"
)

Drive SSH key: /content/drive/MyDrive/hadoop-hdfs.pem
Colab SSH key: /content/hadoop-hdfs.pem
Colab SSH key bytes: 1678
Colab SSH key preparation: PASS


In [3]:
# ============================================================
# CELL 2: REMOTE_EXEC.SH VALIDATION AND REMOTE HELPER
#
# The EC2 host, user, SSH key and SSH options remain inside
# remote_exec.sh. Nothing is duplicated or hardcoded here.
# ============================================================

if not REMOTE_EXEC.is_file():
    raise FileNotFoundError(
        "remote_exec.sh is unavailable: "
        + str(REMOTE_EXEC)
    )

REMOTE_EXEC.chmod(0o700)


def run_remote(
    command,
    timeout=300,
    show_output=True,
):
    result = subprocess.run(
        [
            str(REMOTE_EXEC),
            command,
        ],
        text=True,
        capture_output=True,
        timeout=timeout,
    )

    if show_output and result.stdout:
        print(result.stdout)

    if show_output and result.stderr:
        print("REMOTE STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            "Remote command failed with return code "
            + str(result.returncode)
            + "."
        )

    return result


connection_test = run_remote(
    r'''
set -e

echo "REMOTE_EXEC_CONNECTION_SUCCESS"
echo "Remote user: $(whoami)"
echo "Remote host: $(hostname)"
''',
    timeout=90,
    show_output=True,
)

if (
    "REMOTE_EXEC_CONNECTION_SUCCESS"
    not in connection_test.stdout
):
    raise RuntimeError(
        "remote_exec.sh returned successfully but the "
        "connection marker was not found."
    )

print("remote_exec.sh connection: PASS")
print("Hardcoded EC2 host in notebook: NO")
print("Hardcoded SSH key in notebook: NO")
print("CELL 2: SUCCESS")

REMOTE_EXEC_CONNECTION_SUCCESS
Remote user: ubuntu
Remote host: ip-172-31-30-94

REMOTE STDERR:

remote_exec.sh connection: PASS
Hardcoded EC2 host in notebook: NO
Hardcoded SSH key in notebook: NO
CELL 2: SUCCESS


In [8]:
# ============================================================
# CELL 3: READ-ONLY EC2/HDFS/SPARK/SEDONA PREFLIGHT
# ============================================================

cmd=r'''set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
echo "============================================================"
echo "SIX-CITY MODELING PREFLIGHT"
echo "============================================================"
for daemon in NameNode DataNode SecondaryNameNode
do
  jps | awk '{print $2}' | grep -qx "$daemon"
done
echo "HDFS daemons: PASS"
hdfs dfsadmin -safemode get | grep -qi "Safe mode is OFF"
echo "HDFS safe mode: OFF"
hdfs dfs -test -d /data/ghsl/curated/cities
echo "Curated city root: PASS"
command -v spark-submit >/dev/null
echo "spark-submit: PASS"
python3 - <<'PY'
import importlib.util
missing=[x for x in ["pyspark","sedona"] if importlib.util.find_spec(x) is None]
if missing: raise RuntimeError("Missing packages: "+repr(missing))
print("PySpark import: PASS")
print("Sedona import: PASS")
PY
echo "HDFS modified: NO"
echo "Spark job launched: NO"
echo "SIX_CITY_MODELING_PREFLIGHT_SUCCESS"'''

r=run_remote(cmd,timeout=300,show_output=True)
print("Return code:",r.returncode)
print("CELL 3: SUCCESS")

SIX-CITY MODELING PREFLIGHT
HDFS daemons: PASS
HDFS safe mode: OFF
Curated city root: PASS
spark-submit: PASS
PySpark import: PASS
Sedona import: PASS
HDFS modified: NO
Spark job launched: NO
SIX_CITY_MODELING_PREFLIGHT_SUCCESS

REMOTE STDERR:

Return code: 0
CELL 3: SUCCESS


In [5]:
# ============================================================
# DIAGNOSTIC: IDENTIFY EXACT CELL 3 FAILURE
# ============================================================

cmd = r'''
source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

echo "========== JAVA DAEMONS =========="
jps

echo "========== HDFS SAFE MODE =========="
hdfs dfsadmin -safemode get

echo "========== CURATED CITY ROOT =========="
hdfs dfs -ls /data/ghsl/curated/cities

echo "========== SPARK =========="
command -v spark-submit

echo "========== PYTHON PACKAGES =========="
python3 -c "import pyspark; print('PySpark: PASS')"
python3 -c "import sedona; print('Sedona: PASS')"
'''

result = run_remote(
    cmd,
    timeout=300,
    show_output=True
)

print("CELL 3 DIAGNOSTIC: COMPLETE")

========== JAVA DAEMONS ==========
1359 Jps
========== HDFS SAFE MODE ==========
========== CURATED CITY ROOT ==========
========== SPARK ==========
/home/ubuntu/ghsl-sedona-env/bin/spark-submit
========== PYTHON PACKAGES ==========
PySpark: PASS
Sedona: PASS

REMOTE STDERR:
safemode: Call From ip-172-31-30-94/172.31.30.94 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused
ls: Call From ip-172-31-30-94/172.31.30.94 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused

CELL 3 DIAGNOSTIC: COMPLETE


In [6]:
# ============================================================
# START HADOOP HDFS DAEMONS
# ============================================================

cmd = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

echo "========== STARTING HDFS =========="

start-dfs.sh

echo
echo "========== JAVA DAEMONS =========="
jps

echo
echo "========== HDFS SAFE MODE =========="
hdfs dfsadmin -safemode get

echo
echo "========== CURATED CITY ROOT =========="
hdfs dfs -ls /data/ghsl/curated/cities

echo
echo "HDFS STARTUP CHECK COMPLETE"
'''

r = run_remote(
    cmd,
    timeout=300,
    show_output=True
)

print("Return code:", r.returncode)

========== STARTING HDFS ==========
Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [ip-172-31-30-94]

========== JAVA DAEMONS ==========
1712 NameNode
2228 Jps
1852 DataNode
2093 SecondaryNameNode

========== HDFS SAFE MODE ==========
Safe mode is ON

========== CURATED CITY ROOT ==========
Found 6 items
drwxr-xr-x   - ubuntu supergroup          0 2026-11-09 09:17 /data/ghsl/curated/cities/city=bengaluru
drwxr-xr-x   - ubuntu supergroup          0 2026-11-08 14:45 /data/ghsl/curated/cities/city=chennai_metro
drwxr-xr-x   - ubuntu supergroup          0 2026-11-08 16:37 /data/ghsl/curated/cities/city=delhi_ncr
drwxr-xr-x   - ubuntu supergroup          0 2026-11-07 15:50 /data/ghsl/curated/cities/city=hyderabad
drwxr-xr-x   - ubuntu supergroup          0 2026-11-09 05:08 /data/ghsl/curated/cities/city=mumbai_metro
drwxr-xr-x   - ubuntu supergroup          0 2026-11-08 12:10 /data/ghsl/curated/cities/city=pune_metro

HDFS STARTUP CHECK COMPLETE

REMOTE ST

In [7]:
# ============================================================
# CHECK HDFS READINESS
# ============================================================

cmd = r'''
source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

echo "========== HDFS SAFE MODE =========="
hdfs dfsadmin -safemode get

echo
echo "========== HDFS HEALTH =========="
hdfs dfsadmin -report

echo
echo "========== HDFS ROOT CHECK =========="
hdfs dfs -ls /data/ghsl/curated/cities
'''

r = run_remote(
    cmd,
    timeout=300,
    show_output=True
)

========== HDFS SAFE MODE ==========
Safe mode is OFF

========== HDFS HEALTH ==========
Configured Capacity: 82085269504 (76.45 GB)
Present Capacity: 63991836672 (59.60 GB)
DFS Remaining: 56796815360 (52.90 GB)
DFS Used: 7195021312 (6.70 GB)
DFS Used%: 11.24%
Replicated Blocks:
	Under replicated blocks: 1
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 1
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
Live datanodes (1):

Name: 127.0.0.1:9866 (localhost)
Hostname: ip-172-31-30-94.ec2.internal
Decommission Status : Normal
Configured Capacity: 82085269504 (76.45 GB)
DFS Used: 7195021312 (6.70 GB)
Non DFS Used: 18076655616 (16.84 GB)
DF

In [9]:
# ============================================================
# CELL 4: DISCOVER EXACT HDFS CITY SLUGS AND ALL EPOCH PATHS
#
# Read-only:
#   - Lists curated city folders
#   - Discovers enriched_grid_year partitions
#   - Does not read full Parquet data
#   - Does not modify HDFS
# ============================================================

cmd = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

python3 - <<'PY'
import json
import re
import subprocess


CITY_ROOT = (
    "/data/ghsl/curated/cities"
)

EXPECTED_EPOCHS = [
    1975,
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
    2020,
]


result = subprocess.run(
    [
        "hdfs",
        "dfs",
        "-ls",
        CITY_ROOT,
    ],
    text=True,
    capture_output=True,
)


if result.returncode != 0:
    raise RuntimeError(
        "Could not list the curated city root.\n"
        + result.stderr
    )


city_slugs = []


for line in result.stdout.splitlines():
    fields = line.split()

    if not fields:
        continue

    path = fields[-1]

    match = re.search(
        r"/city=([^/]+)$",
        path,
    )

    if match:
        city_slugs.append(
            match.group(1)
        )


city_slugs = sorted(
    set(
        city_slugs
    )
)


if not city_slugs:
    raise RuntimeError(
        "No city=<slug> directories were found "
        "under "
        + CITY_ROOT
    )


partitions = []


for city_slug in city_slugs:
    enriched_root = (
        CITY_ROOT
        + "/city="
        + city_slug
        + "/enriched_grid_year"
    )

    enriched_exists = (
        subprocess.run(
            [
                "hdfs",
                "dfs",
                "-test",
                "-d",
                enriched_root,
            ]
        ).returncode
        == 0
    )

    if not enriched_exists:
        continue

    for epoch in EXPECTED_EPOCHS:
        epoch_path = (
            enriched_root
            + "/epoch="
            + str(epoch)
        )

        epoch_exists = (
            subprocess.run(
                [
                    "hdfs",
                    "dfs",
                    "-test",
                    "-d",
                    epoch_path,
                ]
            ).returncode
            == 0
        )

        partitions.append(
            {
                "city_slug": city_slug,
                "epoch": epoch,
                "path": (
                    "hdfs://localhost:9000"
                    + epoch_path
                ),
                "exists": epoch_exists,
            }
        )


payload = {
    "all_city_slugs": city_slugs,
    "partitions": partitions,
}


print(
    "INVENTORY_JSON_BEGIN"
)

print(
    json.dumps(
        payload,
        sort_keys=True,
    )
)

print(
    "INVENTORY_JSON_END"
)
PY

echo "HDFS modified: NO"
echo "CELL_4_REMOTE_SUCCESS"
'''


inventory_result = run_remote(
    cmd,
    timeout=300,
    show_output=False,
)


# ============================================================
# PARSE THE REMOTE JSON WITHOUT A MULTILINE REGEX
# ============================================================

begin_marker = (
    "INVENTORY_JSON_BEGIN"
)

end_marker = (
    "INVENTORY_JSON_END"
)


begin_index = (
    inventory_result.stdout.find(
        begin_marker
    )
)

end_index = (
    inventory_result.stdout.find(
        end_marker
    )
)


if begin_index < 0:
    print(
        inventory_result.stdout
    )

    if inventory_result.stderr:
        print(
            "REMOTE STDERR:"
        )

        print(
            inventory_result.stderr
        )

    raise RuntimeError(
        "The inventory begin marker "
        "was not found."
    )


if end_index < 0:
    print(
        inventory_result.stdout
    )

    if inventory_result.stderr:
        print(
            "REMOTE STDERR:"
        )

        print(
            inventory_result.stderr
        )

    raise RuntimeError(
        "The inventory end marker "
        "was not found."
    )


if end_index <= begin_index:
    raise RuntimeError(
        "The inventory markers are "
        "in the wrong order."
    )


json_start = (
    begin_index
    + len(
        begin_marker
    )
)


inventory_json = (
    inventory_result.stdout[
        json_start:
        end_index
    ]
    .strip()
)


if not inventory_json:
    raise RuntimeError(
        "The remote inventory JSON "
        "payload is empty."
    )


try:
    RAW_HDFS_INVENTORY = (
        json.loads(
            inventory_json
        )
    )

except json.JSONDecodeError as error:
    print(
        "Inventory JSON payload:"
    )

    print(
        inventory_json
    )

    raise RuntimeError(
        "Could not parse the HDFS "
        "inventory JSON."
    ) from error


# ============================================================
# VALIDATE THE PARSED INVENTORY
# ============================================================

if (
    "all_city_slugs"
    not in RAW_HDFS_INVENTORY
):
    raise RuntimeError(
        "all_city_slugs is missing "
        "from the inventory."
    )


if (
    "partitions"
    not in RAW_HDFS_INVENTORY
):
    raise RuntimeError(
        "partitions is missing "
        "from the inventory."
    )


DISCOVERED_CITY_SLUGS = (
    RAW_HDFS_INVENTORY[
        "all_city_slugs"
    ]
)


if not isinstance(
    DISCOVERED_CITY_SLUGS,
    list,
):
    raise RuntimeError(
        "all_city_slugs is not a list."
    )


if not DISCOVERED_CITY_SLUGS:
    raise RuntimeError(
        "No HDFS city slugs were discovered."
    )


discovered_partition_records = (
    RAW_HDFS_INVENTORY[
        "partitions"
    ]
)


if not isinstance(
    discovered_partition_records,
    list,
):
    raise RuntimeError(
        "partitions is not a list."
    )


# ============================================================
# REPORT
# ============================================================

print(
    "Discovered HDFS city slugs:"
)

for city_slug in (
    DISCOVERED_CITY_SLUGS
):
    print(
        " -",
        city_slug,
    )


cities_with_enriched_outputs = sorted(
    {
        record["city_slug"]
        for record
        in discovered_partition_records
    }
)


existing_partition_count = sum(
    1
    for record
    in discovered_partition_records
    if record.get("exists") is True
)


print()

print(
    "Cities with enriched outputs:",
    len(
        cities_with_enriched_outputs
    ),
)

print(
    "Existing enriched epoch partitions:",
    existing_partition_count,
)

print(
    "HDFS modified: NO"
)

print(
    "CELL 4: SUCCESS"
)

Discovered HDFS city slugs:
 - bengaluru
 - chennai_metro
 - delhi_ncr
 - hyderabad
 - mumbai_metro
 - pune_metro

Cities with enriched outputs: 6
Existing enriched epoch partitions: 60
HDFS modified: NO
CELL 4: SUCCESS


In [10]:
# ============================================================
# CELL 4A: READ-ONLY BENGALURU HDFS LOCATION AUDIT
#
# Purpose:
#   - Locate Bengaluru outputs stored outside the newer
#     /data/ghsl/curated/cities/city=<slug> structure
#   - Identify enriched_grid_year epoch partitions
#   - Identify possible longitudinal or epoch-summary outputs
#   - Produce machine-readable candidate information
#
# Read-only:
#   - Does not launch Spark
#   - Does not read Parquet contents
#   - Does not create, move, copy, or delete HDFS data
#   - Does not modify local or remote source files
# ============================================================

bengaluru_search_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

echo "============================================================"
echo "BENGALURU HDFS LOCATION AUDIT"
echo "============================================================"


# ============================================================
# REMOTE READ-ONLY DISCOVERY
# ============================================================

python3 - <<'PY'
import json
import re
import subprocess


SEARCH_ROOTS = [
    "/data/ghsl",
    "/user/ubuntu",
]


EXPECTED_EPOCHS = [
    1975,
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
    2020,
]


CITY_TERMS = [
    "bengaluru",
    "bangalore",
]


STRUCTURE_TERMS = [
    "enriched_grid_year",
    "enriched",
    "longitudinal",
    "epoch_summary",
    "epoch-summary",
    "historical",
    "grid_1km",
    "curated",
]


def run_hdfs(
    arguments,
    allow_failure=False,
):
    result = subprocess.run(
        [
            "hdfs",
            "dfs",
            *arguments,
        ],
        text=True,
        capture_output=True,
    )

    if (
        result.returncode != 0
        and not allow_failure
    ):
        raise RuntimeError(
            "HDFS command failed: "
            + repr(
                arguments
            )
            + "\n"
            + result.stderr
        )

    return result


def hdfs_directory_exists(path):
    return (
        run_hdfs(
            [
                "-test",
                "-d",
                path,
            ],
            allow_failure=True,
        ).returncode
        == 0
    )


def hdfs_path_exists(path):
    return (
        run_hdfs(
            [
                "-test",
                "-e",
                path,
            ],
            allow_failure=True,
        ).returncode
        == 0
    )


def normalize_path(line):
    fields = line.split()

    if not fields:
        return None

    candidate = fields[-1]

    if not candidate.startswith("/"):
        return None

    return candidate


all_paths = set()


# ============================================================
# RECURSIVE HDFS NAME DISCOVERY
#
# These are metadata listings only. File contents are not read.
# ============================================================

for search_root in SEARCH_ROOTS:
    if not hdfs_directory_exists(
        search_root
    ):
        continue

    listing = run_hdfs(
        [
            "-ls",
            "-R",
            search_root,
        ],
        allow_failure=True,
    )

    for line in listing.stdout.splitlines():
        path = normalize_path(
            line
        )

        if path is not None:
            all_paths.add(
                path
            )


all_paths = sorted(
    all_paths
)


# ============================================================
# CITY-NAME MATCHES
# ============================================================

city_matches = []

for path in all_paths:
    lowered = path.lower()

    if any(
        term in lowered
        for term in CITY_TERMS
    ):
        city_matches.append(
            path
        )


# ============================================================
# STRUCTURAL MATCHES
#
# Include enriched/longitudinal structures that may lack the
# Bengaluru name but exist under an older prototype root.
# ============================================================

structural_matches = []

for path in all_paths:
    lowered = path.lower()

    if any(
        term in lowered
        for term in STRUCTURE_TERMS
    ):
        structural_matches.append(
            path
        )


# ============================================================
# CANDIDATE ENRICHED ROOTS
# ============================================================

candidate_roots = set()


for path in city_matches:
    if (
        "enriched_grid_year"
        in path.lower()
    ):
        before_epoch = re.sub(
            r"/epoch=\d{4}(?:/.*)?$",
            "",
            path,
        )

        candidate_roots.add(
            before_epoch
        )


for path in structural_matches:
    lowered = path.lower()

    if (
        "enriched_grid_year"
        not in lowered
    ):
        continue

    if any(
        term in lowered
        for term in CITY_TERMS
    ):
        before_epoch = re.sub(
            r"/epoch=\d{4}(?:/.*)?$",
            "",
            path,
        )

        candidate_roots.add(
            before_epoch
        )


# ============================================================
# ADD COMMON LEGACY PROTOTYPE LOCATIONS
#
# Existence checks only. No locations are assumed to be valid
# unless HDFS confirms them.
# ============================================================

common_legacy_roots = [
    (
        "/data/ghsl/curated/"
        "bengaluru/enriched_grid_year"
    ),
    (
        "/data/ghsl/curated/"
        "bangalore/enriched_grid_year"
    ),
    (
        "/data/ghsl/curated/"
        "bengaluru_historical/"
        "enriched_grid_year"
    ),
    (
        "/data/ghsl/curated/"
        "bangalore_historical/"
        "enriched_grid_year"
    ),
    (
        "/data/ghsl/curated/"
        "cities/bengaluru/"
        "enriched_grid_year"
    ),
    (
        "/data/ghsl/curated/"
        "cities/bangalore/"
        "enriched_grid_year"
    ),
    (
        "/data/ghsl/curated/"
        "cities/city=bengaluru/"
        "enriched_grid_year"
    ),
    (
        "/data/ghsl/curated/"
        "cities/city=bangalore/"
        "enriched_grid_year"
    ),
    (
        "/data/ghsl/curated/"
        "enriched_grid_year"
    ),
]


for root in common_legacy_roots:
    if hdfs_directory_exists(
        root
    ):
        candidate_roots.add(
            root
        )


candidate_roots = sorted(
    candidate_roots
)


# ============================================================
# VERIFY EPOCH COVERAGE FOR EACH CANDIDATE ROOT
# ============================================================

candidate_reports = []


for root in candidate_roots:
    epochs_found = []
    epoch_paths = {}

    for epoch in EXPECTED_EPOCHS:
        epoch_path = (
            root
            + "/epoch="
            + str(epoch)
        )

        if hdfs_directory_exists(
            epoch_path
        ):
            epochs_found.append(
                epoch
            )

            epoch_paths[
                str(epoch)
            ] = (
                "hdfs://localhost:9000"
                + epoch_path
            )

    candidate_reports.append(
        {
            "root": (
                "hdfs://localhost:9000"
                + root
            ),
            "raw_root": root,
            "epochs_found": epochs_found,
            "epoch_count": len(
                epochs_found
            ),
            "complete_10_epoch_root": (
                epochs_found
                == EXPECTED_EPOCHS
            ),
            "epoch_paths": epoch_paths,
        }
    )


candidate_reports = sorted(
    candidate_reports,
    key=lambda record: (
        -record["epoch_count"],
        record["raw_root"],
    ),
)


# ============================================================
# IDENTIFY OTHER BENGALURU ANALYTICS OUTPUTS
# ============================================================

other_bengaluru_outputs = []

for path in city_matches:
    lowered = path.lower()

    if any(
        term in lowered
        for term in [
            "longitudinal",
            "epoch_summary",
            "epoch-summary",
            "transition",
            "historical",
        ]
    ):
        other_bengaluru_outputs.append(
            path
        )


other_bengaluru_outputs = sorted(
    set(
        other_bengaluru_outputs
    )
)


# ============================================================
# BUILD RESULT
# ============================================================

complete_candidates = [
    report
    for report
    in candidate_reports
    if report[
        "complete_10_epoch_root"
    ]
]


result_payload = {
    "search_roots": SEARCH_ROOTS,
    "expected_epochs": EXPECTED_EPOCHS,
    "city_name_matches": city_matches,
    "candidate_enriched_roots": (
        candidate_reports
    ),
    "complete_candidates": (
        complete_candidates
    ),
    "other_bengaluru_outputs": (
        other_bengaluru_outputs
    ),
}


print(
    "BENGALURU_AUDIT_JSON_BEGIN"
)

print(
    json.dumps(
        result_payload,
        sort_keys=True,
    )
)

print(
    "BENGALURU_AUDIT_JSON_END"
)
PY


echo
echo "HDFS modified: NO"
echo "Spark job launched: NO"
echo "BENGALURU_HDFS_LOCATION_AUDIT_COMPLETE"
'''


bengaluru_search_result = run_remote(
    bengaluru_search_command,
    timeout=900,
    show_output=False,
)


# ============================================================
# PARSE RESULT WITHOUT MULTILINE REGULAR EXPRESSIONS
# ============================================================

begin_marker = (
    "BENGALURU_AUDIT_JSON_BEGIN"
)

end_marker = (
    "BENGALURU_AUDIT_JSON_END"
)


begin_index = (
    bengaluru_search_result.stdout.find(
        begin_marker
    )
)

end_index = (
    bengaluru_search_result.stdout.find(
        end_marker
    )
)


if begin_index < 0:
    print(
        bengaluru_search_result.stdout
    )

    if bengaluru_search_result.stderr:
        print(
            "REMOTE STDERR:"
        )

        print(
            bengaluru_search_result.stderr
        )

    raise RuntimeError(
        "The Bengaluru audit begin marker "
        "was not found."
    )


if end_index < 0:
    print(
        bengaluru_search_result.stdout
    )

    if bengaluru_search_result.stderr:
        print(
            "REMOTE STDERR:"
        )

        print(
            bengaluru_search_result.stderr
        )

    raise RuntimeError(
        "The Bengaluru audit end marker "
        "was not found."
    )


if end_index <= begin_index:
    raise RuntimeError(
        "The Bengaluru audit markers "
        "are in the wrong order."
    )


json_start = (
    begin_index
    + len(
        begin_marker
    )
)


audit_json = (
    bengaluru_search_result.stdout[
        json_start:
        end_index
    ]
    .strip()
)


if not audit_json:
    raise RuntimeError(
        "The Bengaluru audit JSON "
        "payload is empty."
    )


try:
    BENGALURU_HDFS_AUDIT = (
        json.loads(
            audit_json
        )
    )

except json.JSONDecodeError as error:
    print(
        "Bengaluru audit JSON:"
    )

    print(
        audit_json
    )

    raise RuntimeError(
        "Could not parse the Bengaluru "
        "HDFS audit JSON."
    ) from error


# ============================================================
# REPORT RESULTS
# ============================================================

city_name_matches = (
    BENGALURU_HDFS_AUDIT[
        "city_name_matches"
    ]
)

candidate_enriched_roots = (
    BENGALURU_HDFS_AUDIT[
        "candidate_enriched_roots"
    ]
)

complete_candidates = (
    BENGALURU_HDFS_AUDIT[
        "complete_candidates"
    ]
)

other_outputs = (
    BENGALURU_HDFS_AUDIT[
        "other_bengaluru_outputs"
    ]
)


print(
    "Bengaluru/Bangalore HDFS name matches:",
    len(
        city_name_matches
    ),
)


if city_name_matches:
    print()

    print(
        "Representative name matches:"
    )

    for path in city_name_matches[:50]:
        print(
            " -",
            path,
        )

    if len(city_name_matches) > 50:
        print(
            " - ...",
            len(city_name_matches) - 50,
            "additional matches",
        )


print()

print(
    "Candidate enriched roots:",
    len(
        candidate_enriched_roots
    ),
)


for report in candidate_enriched_roots:
    print()

    print(
        "Root:",
        report["root"],
    )

    print(
        "Epochs found:",
        report["epochs_found"],
    )

    print(
        "Epoch count:",
        report["epoch_count"],
    )

    print(
        "Complete 10-epoch root:",
        report[
            "complete_10_epoch_root"
        ],
    )


print()

print(
    "Complete Bengaluru candidates:",
    len(
        complete_candidates
    ),
)


for report in complete_candidates:
    print(
        " -",
        report["root"],
    )


if other_outputs:
    print()

    print(
        "Other Bengaluru analytics outputs:"
    )

    for path in other_outputs[:50]:
        print(
            " -",
            path,
        )

    if len(other_outputs) > 50:
        print(
            " - ...",
            len(other_outputs) - 50,
            "additional outputs",
        )


print()

print(
    "HDFS modified: NO"
)

print(
    "Spark job launched: NO"
)

print(
    "CELL 4A: SUCCESS"
)

Bengaluru/Bangalore HDFS name matches: 660

Representative name matches:
 - /data/ghsl/analytics/pilots/bengaluru
 - /data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020
 - /data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020/cell_change
 - /data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020/cell_change/_SUCCESS
 - /data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020/cell_change/part-00000-325271be-9d33-44d4-9f3f-e76c4b2e026e-c000.snappy.parquet
 - /data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020/cell_change/part-00001-325271be-9d33-44d4-9f3f-e76c4b2e026e-c000.snappy.parquet
 - /data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020/cell_change/part-00002-325271be-9d33-44d4-9f3f-e76c4b2e026e-c000.snappy.parquet
 - /data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020/cell_change/part-00003-325271be-9d33-44d4-9f3f-e76c4b2e026e-c000.snappy.parquet
 - /data/ghsl/analytics/pilots/bengaluru/integrate

The exact validated Bengaluru enriched root is:

Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/enriched_grid_year

It contains all ten required epochs:

Plain Text
1975, 1980, 1985, 1990, 1995,
2000, 2005, 2010, 2015, 2020

Therefore:

Plain Text
Training cities: 6/6 found
Expected enriched partitions: 60/60 available
Bengaluru rebuild required: NO
HDFS migration required: NO

In [ ]:
We have two complete Bengaluru dataset roots:

1. Standard city directory

Plain Text
hdfs://localhost:9000/data/ghsl/curated/cities/city=bengaluru/enriched_grid_year

2. Legacy pilot directory

Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/enriched_grid_year

In [12]:
# ============================================================
# CELL 5: RESOLVE THE SIX-CITY TRAINING INVENTORY
#
# Bengaluru uses its standardized city layout:
#
#   /data/ghsl/curated/cities/city=bengaluru/
#
# The other five cities use the generalized city layout:
#
#   /data/ghsl/curated/cities/city=<slug>/
#
# Read-only:
#   - Does not move or copy Bengaluru data
#   - Does not modify HDFS
#   - Does not launch Spark
# ============================================================

if (
    "RAW_HDFS_INVENTORY"
    not in globals()
):
    raise RuntimeError(
        "RAW_HDFS_INVENTORY is unavailable. "
        "Run corrected Cell 4 first."
    )


if (
    "BENGALURU_HDFS_AUDIT"
    not in globals()
):
    raise RuntimeError(
        "BENGALURU_HDFS_AUDIT is unavailable. "
        "Run Cell 4A first."
    )


# ============================================================
# SELECT THE STANDARDIZED BENGALURU CANDIDATE
#
# Both the legacy pilot and standardized city roots exist.
# Prefer the standardized root for six-city modeling.
# ============================================================

complete_bengaluru_candidates = (
    BENGALURU_HDFS_AUDIT.get(
        "complete_candidates",
        [],
    )
)


expected_bengaluru_root = (
    "hdfs://localhost:9000"
    "/data/ghsl/curated/cities/"
    "city=bengaluru/enriched_grid_year"
)


matching_bengaluru_candidates = [
    candidate
    for candidate
    in complete_bengaluru_candidates
    if candidate["root"]
    == expected_bengaluru_root
]


if (
    len(
        matching_bengaluru_candidates
    )
    != 1
):
    raise RuntimeError(
        "Expected exactly one complete standardized "
        "Bengaluru enriched root, found "
        + str(
            len(
                matching_bengaluru_candidates
            )
        )
        + ": "
        + repr(
            matching_bengaluru_candidates
        )
    )


bengaluru_candidate = (
    matching_bengaluru_candidates[0]
)


BENGALURU_ENRICHED_ROOT = (
    bengaluru_candidate["root"]
)


if (
    BENGALURU_ENRICHED_ROOT
    != expected_bengaluru_root
):
    raise RuntimeError(
        "Unexpected Bengaluru enriched root: "
        + BENGALURU_ENRICHED_ROOT
    )


bengaluru_epochs_found = [
    int(epoch)
    for epoch
    in bengaluru_candidate[
        "epochs_found"
    ]
]


if (
    bengaluru_epochs_found
    != EXPECTED_EPOCHS
):
    raise RuntimeError(
        "Bengaluru epoch coverage is incomplete. "
        "Expected "
        + repr(
            EXPECTED_EPOCHS
        )
        + ", found "
        + repr(
            bengaluru_epochs_found
        )
    )


# ============================================================
# RESOLVE THE FIVE GENERALIZED CITY SLUGS
# ============================================================

GENERALIZED_CITY_HINTS = {
    "Hyderabad": [
        "hyderabad",
    ],
    "Pune Metropolitan Region": [
        "pune",
    ],
    "Chennai Metropolitan Region": [
        "chennai",
    ],
    "Mumbai Metropolitan Region": [
        "mumbai",
    ],
    "Delhi NCR": [
        "delhi",
        "ncr",
    ],
}


def resolve_generalized_city_slug(
    city_name,
    hints,
    discovered_slugs,
):
    matches = [
        city_slug
        for city_slug
        in discovered_slugs
        if any(
            hint.lower()
            in city_slug.lower()
            for hint in hints
        )
    ]

    if len(matches) != 1:
        raise RuntimeError(
            "Could not uniquely resolve "
            + city_name
            + ". Hints="
            + repr(hints)
            + ", matches="
            + repr(matches)
            + ", discovered="
            + repr(
                discovered_slugs
            )
        )

    return matches[0]


RESOLVED_GENERALIZED_SLUGS = {}


for (
    city_name,
    city_hints,
) in GENERALIZED_CITY_HINTS.items():
    RESOLVED_GENERALIZED_SLUGS[
        city_name
    ] = (
        resolve_generalized_city_slug(
            city_name,
            city_hints,
            DISCOVERED_CITY_SLUGS,
        )
    )


if (
    len(
        set(
            RESOLVED_GENERALIZED_SLUGS.values()
        )
    )
    != 5
):
    raise RuntimeError(
        "The generalized city slugs are "
        "not unique: "
        + repr(
            RESOLVED_GENERALIZED_SLUGS
        )
    )


# ============================================================
# CREATE EXPLICIT PER-CITY ENRICHED-ROOT MAPPING
#
# All six cities use their standardized city directories.
# Bengaluru's legacy pilot data remains untouched.
# ============================================================

TRAINING_CITY_INPUTS = {
    "Bengaluru": {
        "city_slug": "bengaluru",
        "layout": "generalized_city",
        "enriched_root": (
            BENGALURU_ENRICHED_ROOT
        ),
    },
}


for (
    city_name,
    city_slug,
) in RESOLVED_GENERALIZED_SLUGS.items():
    TRAINING_CITY_INPUTS[
        city_name
    ] = {
        "city_slug": city_slug,
        "layout": (
            "generalized_city"
        ),
        "enriched_root": (
            "hdfs://localhost:9000"
            "/data/ghsl/curated/cities/"
            "city="
            + city_slug
            + "/enriched_grid_year"
        ),
    }


if (
    len(
        TRAINING_CITY_INPUTS
    )
    != EXPECTED_CITY_COUNT
):
    raise RuntimeError(
        "Expected six training-city inputs, "
        "found "
        + str(
            len(
                TRAINING_CITY_INPUTS
            )
        )
    )


if (
    len(
        {
            config["city_slug"]
            for config
            in TRAINING_CITY_INPUTS.values()
        }
    )
    != EXPECTED_CITY_COUNT
):
    raise RuntimeError(
        "Training city slugs are not unique."
    )


# ============================================================
# BUILD THE EXACT 60-PARTITION INVENTORY
# ============================================================

SIX_CITY_PARTITIONS = []


for (
    city_name,
    city_config,
) in TRAINING_CITY_INPUTS.items():
    city_slug = (
        city_config[
            "city_slug"
        ]
    )

    enriched_root = (
        city_config[
            "enriched_root"
        ]
    )

    layout = (
        city_config[
            "layout"
        ]
    )

    for epoch in EXPECTED_EPOCHS:
        epoch_path = (
            enriched_root
            + "/epoch="
            + str(epoch)
        )

        SIX_CITY_PARTITIONS.append(
            {
                "city_name": city_name,
                "city_slug": city_slug,
                "layout": layout,
                "epoch": epoch,
                "path": epoch_path,
            }
        )


if (
    len(
        SIX_CITY_PARTITIONS
    )
    != EXPECTED_PARTITION_COUNT
):
    raise RuntimeError(
        "Expected "
        + str(
            EXPECTED_PARTITION_COUNT
        )
        + " six-city partitions, found "
        + str(
            len(
                SIX_CITY_PARTITIONS
            )
        )
    )


partition_keys = [
    (
        record["city_slug"],
        record["epoch"],
    )
    for record
    in SIX_CITY_PARTITIONS
]


if (
    len(
        set(
            partition_keys
        )
    )
    != EXPECTED_PARTITION_COUNT
):
    raise RuntimeError(
        "Duplicate city-epoch partition "
        "keys were generated."
    )


# ============================================================
# REPORT
# ============================================================

print(
    "Resolved six-city training inventory:"
)


for (
    city_name,
    city_config,
) in TRAINING_CITY_INPUTS.items():
    print()

    print(
        "City:",
        city_name,
    )

    print(
        "Model city slug:",
        city_config[
            "city_slug"
        ],
    )

    print(
        "HDFS layout:",
        city_config[
            "layout"
        ],
    )

    print(
        "Enriched root:",
        city_config[
            "enriched_root"
        ],
    )


print()

print(
    "Training cities:",
    len(
        TRAINING_CITY_INPUTS
    ),
    "/",
    EXPECTED_CITY_COUNT,
)

print(
    "Constructed epoch partitions:",
    len(
        SIX_CITY_PARTITIONS
    ),
    "/",
    EXPECTED_PARTITION_COUNT,
)

print(
    "Bengaluru standardized outputs selected: YES"
)

print(
    "Bengaluru legacy outputs preserved: YES"
)

print(
    "Bengaluru HDFS migration required: NO"
)

print(
    "Jaipur included in training: NO"
)

print(
    "HDFS modified: NO"
)

print(
    "Spark job launched: NO"
)

print(
    "CELL 5: SUCCESS"
)

Resolved six-city training inventory:

City: Bengaluru
Model city slug: bengaluru
HDFS layout: generalized_city
Enriched root: hdfs://localhost:9000/data/ghsl/curated/cities/city=bengaluru/enriched_grid_year

City: Hyderabad
Model city slug: hyderabad
HDFS layout: generalized_city
Enriched root: hdfs://localhost:9000/data/ghsl/curated/cities/city=hyderabad/enriched_grid_year

City: Pune Metropolitan Region
Model city slug: pune_metro
HDFS layout: generalized_city
Enriched root: hdfs://localhost:9000/data/ghsl/curated/cities/city=pune_metro/enriched_grid_year

City: Chennai Metropolitan Region
Model city slug: chennai_metro
HDFS layout: generalized_city
Enriched root: hdfs://localhost:9000/data/ghsl/curated/cities/city=chennai_metro/enriched_grid_year

City: Mumbai Metropolitan Region
Model city slug: mumbai_metro
HDFS layout: generalized_city
Enriched root: hdfs://localhost:9000/data/ghsl/curated/cities/city=mumbai_metro/enriched_grid_year

City: Delhi NCR
Model city slug: delhi_ncr
HD

In [14]:
# ============================================================
# CELL 6: ONE-JOB SPARK AUDIT OF ROWS, KEYS AND SCHEMAS
#
# Audits:
#   - 6 training cities
#   - 10 epochs per city
#   - 60 enriched Parquet partitions
#   - 4,900 rows per partition
#   - 4,900 unique non-null grid_cell_id values per partition
#   - 294,000 total grid-year rows
#   - schema and coverage-column variations
#
# Supports:
#   - Six standardized generalized-city layouts
#   - Bengaluru legacy pilot outputs remain preserved but unused
#
# Read-only:
#   - Does not modify HDFS
#   - Does not modify validated city outputs
#   - Does not write a combined dataset
#   - Removes temporary remote audit files
# ============================================================

for required_name in [
    "SIX_CITY_PARTITIONS",
    "EXPECTED_PARTITION_COUNT",
    "EXPECTED_TOTAL_ROWS",
    "EXPECTED_ROWS_PER_PARTITION",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            required_name
            + " is unavailable. Run Cells 1-5 first."
        )


if (
    len(SIX_CITY_PARTITIONS)
    != EXPECTED_PARTITION_COUNT
):
    raise RuntimeError(
        "Expected "
        + str(EXPECTED_PARTITION_COUNT)
        + " input partition records, found "
        + str(
            len(SIX_CITY_PARTITIONS)
        )
    )


# ============================================================
# ENCODE THE EXACT SIX-CITY PARTITION INVENTORY
# ============================================================

payload_json = json.dumps(
    SIX_CITY_PARTITIONS,
    sort_keys=True,
)

payload_b64 = base64.b64encode(
    payload_json.encode("utf-8")
).decode("ascii")


# ============================================================
# REMOTE SPARK AUDIT
# ============================================================

remote_audit = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

PAYLOAD="/tmp/six_city_inventory_payload.json"
SCRIPT="/tmp/six_city_inventory_audit.py"


echo "__PAYLOAD_B64__" \
| base64 -d \
> "$PAYLOAD"


cat > "$SCRIPT" <<'PY'
from pyspark.sql import (
    SparkSession,
    functions as F,
)

import json
import sys


payload_path = sys.argv[1]


with open(
    payload_path,
    "r",
    encoding="utf-8",
) as handle:
    records = json.load(
        handle
    )


EXPECTED_PARTITIONS = 60
EXPECTED_ROWS_PER_PARTITION = 4900
EXPECTED_TOTAL_ROWS = 294000


if len(records) != EXPECTED_PARTITIONS:
    raise RuntimeError(
        "Expected "
        + str(EXPECTED_PARTITIONS)
        + " partition records, found "
        + str(len(records))
    )


required_record_fields = {
    "city_name",
    "city_slug",
    "layout",
    "epoch",
    "path",
}


for record_index, record in enumerate(
    records
):
    missing_record_fields = sorted(
        required_record_fields
        - set(record)
    )

    if missing_record_fields:
        raise RuntimeError(
            "Partition record "
            + str(record_index)
            + " is missing fields: "
            + repr(
                missing_record_fields
            )
        )


partition_keys = [
    (
        record["city_slug"],
        int(record["epoch"]),
    )
    for record in records
]


if (
    len(
        set(
            partition_keys
        )
    )
    != EXPECTED_PARTITIONS
):
    raise RuntimeError(
        "Duplicate city-epoch records exist "
        "in the audit payload."
    )


spark = (
    SparkSession.builder
    .appName(
        "GHSL-Six-City-Inventory-Audit"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel(
    "WARN"
)


results = []
schema_groups = {}
total_rows = 0


try:
    for item in records:
        city_name = item[
            "city_name"
        ]

        city_slug = item[
            "city_slug"
        ]

        layout = item[
            "layout"
        ]

        epoch = int(
            item["epoch"]
        )

        path = item[
            "path"
        ]


        # ====================================================
        # READ ONE ENRICHED CITY-EPOCH PARTITION
        # ====================================================

        dataframe = (
            spark.read
            .parquet(
                path
            )
        )


        columns = list(
            dataframe.columns
        )


        if (
            "grid_cell_id"
            not in columns
        ):
            raise RuntimeError(
                city_slug
                + " "
                + str(epoch)
                + ": grid_cell_id is missing."
            )


        # ====================================================
        # ROW AND KEY AUDIT
        # ====================================================

        row_count = (
            dataframe.count()
        )

        distinct_grid_cell_count = (
            dataframe
            .select(
                "grid_cell_id"
            )
            .distinct()
            .count()
        )

        null_grid_cell_count = (
            dataframe
            .filter(
                F.col(
                    "grid_cell_id"
                ).isNull()
            )
            .count()
        )


        if (
            row_count
            != EXPECTED_ROWS_PER_PARTITION
        ):
            raise RuntimeError(
                city_slug
                + " "
                + str(epoch)
                + ": expected "
                + str(
                    EXPECTED_ROWS_PER_PARTITION
                )
                + " rows, found "
                + str(row_count)
                + "."
            )


        if (
            distinct_grid_cell_count
            != EXPECTED_ROWS_PER_PARTITION
        ):
            raise RuntimeError(
                city_slug
                + " "
                + str(epoch)
                + ": expected "
                + str(
                    EXPECTED_ROWS_PER_PARTITION
                )
                + " distinct grid_cell_id values, found "
                + str(
                    distinct_grid_cell_count
                )
                + "."
            )


        if null_grid_cell_count != 0:
            raise RuntimeError(
                city_slug
                + " "
                + str(epoch)
                + ": found "
                + str(
                    null_grid_cell_count
                )
                + " null grid_cell_id values."
            )


        # ====================================================
        # SCHEMA CAPTURE
        # ====================================================

        schema_signature = [
            {
                "name": field.name,
                "type": (
                    field.dataType
                    .simpleString()
                ),
                "nullable": (
                    field.nullable
                ),
            }
            for field
            in dataframe.schema.fields
        ]


        schema_key = json.dumps(
            schema_signature,
            sort_keys=True,
        )


        schema_groups.setdefault(
            schema_key,
            [],
        ).append(
            city_slug
            + ":"
            + str(epoch)
        )


        coverage_columns = sorted(
            column
            for column
            in columns
            if (
                "coverage"
                in column.lower()
            )
        )


        results.append(
            {
                "city_name": city_name,
                "city_slug": city_slug,
                "layout": layout,
                "epoch": epoch,
                "path": path,
                "row_count": row_count,
                "distinct_grid_cell_id": (
                    distinct_grid_cell_count
                ),
                "null_grid_cell_id": (
                    null_grid_cell_count
                ),
                "columns": columns,
                "coverage_columns": (
                    coverage_columns
                ),
                "schema_signature": (
                    schema_signature
                ),
            }
        )


        total_rows += row_count


        print(
            "PASS:",
            city_slug,
            epoch,
            "layout="
            + layout,
            "rows="
            + str(row_count),
            "keys="
            + str(
                distinct_grid_cell_count
            ),
        )


    # ========================================================
    # GLOBAL AUDIT
    # ========================================================

    if len(results) != EXPECTED_PARTITIONS:
        raise RuntimeError(
            "Expected "
            + str(
                EXPECTED_PARTITIONS
            )
            + " audited partitions, found "
            + str(
                len(results)
            )
            + "."
        )


    if total_rows != EXPECTED_TOTAL_ROWS:
        raise RuntimeError(
            "Expected "
            + str(
                EXPECTED_TOTAL_ROWS
            )
            + " total rows, found "
            + str(total_rows)
            + "."
        )


    audited_city_slugs = sorted(
        {
            record["city_slug"]
            for record
            in results
        }
    )


    if len(audited_city_slugs) != 6:
        raise RuntimeError(
            "Expected six audited city slugs, found "
            + repr(
                audited_city_slugs
            )
        )


    bengaluru_partition_count = sum(
        1
        for record
        in results
        if (
            record["city_slug"]
            == "bengaluru"
        )
    )


    generalized_partition_count = sum(
        1
        for record
        in results
        if (
            record["layout"]
            == "generalized_city"
        )
    )


    unexpected_layouts = sorted(
        {
            record["layout"]
            for record
            in results
            if (
                record["layout"]
                != "generalized_city"
            )
        }
    )


    if bengaluru_partition_count != 10:
        raise RuntimeError(
            "Expected 10 Bengaluru partitions, "
            "found "
            + str(
                bengaluru_partition_count
            )
            + "."
        )


    if generalized_partition_count != 60:
        raise RuntimeError(
            "Expected 60 generalized-city partitions, "
            "found "
            + str(
                generalized_partition_count
            )
            + "."
        )


    if unexpected_layouts:
        raise RuntimeError(
            "Unexpected HDFS layouts found: "
            + repr(
                unexpected_layouts
            )
        )


    output = {
        "partition_count": (
            len(results)
        ),
        "total_rows": total_rows,
        "audited_city_slugs": (
            audited_city_slugs
        ),
        "bengaluru_partition_count": (
            bengaluru_partition_count
        ),
        "generalized_partition_count": (
            generalized_partition_count
        ),
        "unexpected_layouts": (
            unexpected_layouts
        ),
        "schema_variant_count": (
            len(
                schema_groups
            )
        ),
        "schema_groups": list(
            schema_groups.values()
        ),
        "partitions": results,
    }


    print(
        "AUDIT_JSON_BEGIN"
    )

    print(
        json.dumps(
            output,
            sort_keys=True,
        )
    )

    print(
        "AUDIT_JSON_END"
    )

    print(
        "SIX_CITY_ROW_AND_KEY_AUDIT_SUCCESS"
    )


finally:
    spark.stop()
PY


spark-submit \
    --master local[1] \
    --driver-memory 4g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=4 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=512m \
    "$SCRIPT" \
    "$PAYLOAD"


rm -f \
    "$SCRIPT" \
    "$PAYLOAD"


echo
echo "Temporary audit files removed: PASS"
echo "HDFS modified: NO"
echo "SIX_CITY_REMOTE_AUDIT_COMPLETE"
'''


remote_audit = (
    remote_audit.replace(
        "__PAYLOAD_B64__",
        payload_b64,
    )
)


audit_result = run_remote(
    remote_audit,
    timeout=3600,
    show_output=True,
)


# ============================================================
# PARSE REMOTE AUDIT JSON WITHOUT A MULTILINE REGEX
# ============================================================

begin_marker = (
    "AUDIT_JSON_BEGIN"
)

end_marker = (
    "AUDIT_JSON_END"
)


begin_index = (
    audit_result.stdout.find(
        begin_marker
    )
)

end_index = (
    audit_result.stdout.find(
        end_marker
    )
)


if begin_index < 0:
    print(
        audit_result.stdout[
            -12000:
        ]
    )

    if audit_result.stderr:
        print(
            "REMOTE STDERR:"
        )

        print(
            audit_result.stderr[
                -4000:
            ]
        )

    raise RuntimeError(
        "The audit JSON begin marker "
        "was not found."
    )


if end_index < 0:
    print(
        audit_result.stdout[
            -12000:
        ]
    )

    if audit_result.stderr:
        print(
            "REMOTE STDERR:"
        )

        print(
            audit_result.stderr[
                -4000:
            ]
        )

    raise RuntimeError(
        "The audit JSON end marker "
        "was not found."
    )


if end_index <= begin_index:
    raise RuntimeError(
        "The audit JSON markers are "
        "in the wrong order."
    )


json_start = (
    begin_index
    + len(
        begin_marker
    )
)


audit_json = (
    audit_result.stdout[
        json_start:
        end_index
    ]
    .strip()
)


if not audit_json:
    raise RuntimeError(
        "The audit JSON payload is empty."
    )


try:
    SIX_CITY_AUDIT = json.loads(
        audit_json
    )

except json.JSONDecodeError as error:
    print(
        "Audit JSON payload:"
    )

    print(
        audit_json[
            :12000
        ]
    )

    raise RuntimeError(
        "Could not parse the Spark "
        "audit JSON."
    ) from error


# ============================================================
# VALIDATE PARSED AUDIT RESULT
# ============================================================

if (
    SIX_CITY_AUDIT.get(
        "partition_count"
    )
    != EXPECTED_PARTITION_COUNT
):
    raise RuntimeError(
        "Expected "
        + str(
            EXPECTED_PARTITION_COUNT
        )
        + " audited partitions, found "
        + str(
            SIX_CITY_AUDIT.get(
                "partition_count"
            )
        )
    )


if (
    SIX_CITY_AUDIT.get(
        "total_rows"
    )
    != EXPECTED_TOTAL_ROWS
):
    raise RuntimeError(
        "Expected "
        + str(
            EXPECTED_TOTAL_ROWS
        )
        + " total rows, found "
        + str(
            SIX_CITY_AUDIT.get(
                "total_rows"
            )
        )
    )


partition_results = (
    SIX_CITY_AUDIT.get(
        "partitions",
        [],
    )
)


if (
    len(
        partition_results
    )
    != EXPECTED_PARTITION_COUNT
):
    raise RuntimeError(
        "The audit partition-result count "
        "is incorrect."
    )


invalid_partitions = [
    {
        "city_slug": record.get(
            "city_slug"
        ),
        "epoch": record.get(
            "epoch"
        ),
        "row_count": record.get(
            "row_count"
        ),
        "distinct_grid_cell_id": (
            record.get(
                "distinct_grid_cell_id"
            )
        ),
        "null_grid_cell_id": (
            record.get(
                "null_grid_cell_id"
            )
        ),
    }
    for record
    in partition_results
    if (
        record.get(
            "row_count"
        )
        != EXPECTED_ROWS_PER_PARTITION
        or record.get(
            "distinct_grid_cell_id"
        )
        != EXPECTED_ROWS_PER_PARTITION
        or record.get(
            "null_grid_cell_id"
        )
        != 0
    )
]


if invalid_partitions:
    raise RuntimeError(
        "One or more audited partitions "
        "failed row/key validation: "
        + repr(
            invalid_partitions
        )
    )


if (
    SIX_CITY_AUDIT.get(
        "bengaluru_partition_count"
    )
    != 10
):
    raise RuntimeError(
        "Expected 10 standardized Bengaluru "
        "partitions."
    )


if (
    SIX_CITY_AUDIT.get(
        "generalized_partition_count"
    )
    != 60
):
    raise RuntimeError(
        "Expected 60 generalized-city "
        "partitions."
    )


if SIX_CITY_AUDIT.get(
    "unexpected_layouts",
    [],
):
    raise RuntimeError(
        "Unexpected HDFS layouts were audited: "
        + repr(
            SIX_CITY_AUDIT.get(
                "unexpected_layouts"
            )
        )
    )


# ============================================================
# REPORT
# ============================================================

print(
    "Partitions audited:",
    SIX_CITY_AUDIT[
        "partition_count"
    ],
)

print(
    "Total rows:",
    SIX_CITY_AUDIT[
        "total_rows"
    ],
)

print(
    "Audited city slugs:",
    SIX_CITY_AUDIT[
        "audited_city_slugs"
    ],
)

print(
    "Bengaluru standardized partitions:",
    SIX_CITY_AUDIT[
        "bengaluru_partition_count"
    ],
)

print(
    "Generalized-city partitions:",
    SIX_CITY_AUDIT[
        "generalized_partition_count"
    ],
)

print(
    "Unexpected HDFS layouts:",
    SIX_CITY_AUDIT[
        "unexpected_layouts"
    ],
)

print(
    "Bengaluru legacy outputs used in training: NO"
)

print(
    "Bengaluru legacy outputs preserved: YES"
)

print(
    "Schema variants:",
    SIX_CITY_AUDIT[
        "schema_variant_count"
    ],
)

print(
    "Duplicate grid keys: 0"
)

print(
    "Null grid keys: 0"
)

print(
    "Jaipur included in training: NO"
)

print(
    "Temporary audit files removed: PASS"
)

print(
    "HDFS modified: NO"
)

print(
    "CELL 6: SUCCESS"
)

PASS: bengaluru 1975 layout=generalized_city rows=4900 keys=4900
PASS: bengaluru 1980 layout=generalized_city rows=4900 keys=4900
PASS: bengaluru 1985 layout=generalized_city rows=4900 keys=4900
PASS: bengaluru 1990 layout=generalized_city rows=4900 keys=4900
PASS: bengaluru 1995 layout=generalized_city rows=4900 keys=4900
PASS: bengaluru 2000 layout=generalized_city rows=4900 keys=4900
PASS: bengaluru 2005 layout=generalized_city rows=4900 keys=4900
PASS: bengaluru 2010 layout=generalized_city rows=4900 keys=4900
PASS: bengaluru 2015 layout=generalized_city rows=4900 keys=4900
PASS: bengaluru 2020 layout=generalized_city rows=4900 keys=4900
PASS: hyderabad 1975 layout=generalized_city rows=4900 keys=4900
PASS: hyderabad 1980 layout=generalized_city rows=4900 keys=4900
PASS: hyderabad 1985 layout=generalized_city rows=4900 keys=4900
PASS: hyderabad 1990 layout=generalized_city rows=4900 keys=4900
PASS: hyderabad 1995 layout=generalized_city rows=4900 keys=4900
PASS: hyderabad 2000 layo

In [15]:
# ============================================================
# CELL 7: SCHEMA/COVERAGE REPORT AND LOCAL INVENTORY FREEZE
#
# Creates a local JSON inventory describing:
#   - The six validated training cities
#   - The held-out Jaipur generalization city
#   - The validated 60-partition, 294,000-row inventory
#   - Columns available across all six cities
#   - Coverage columns available by city
#   - Schema variants requiring harmonization
#
# Local-only:
#   - Writes one JSON report under LOCAL_ARTIFACT_ROOT
#   - Does not modify HDFS
#   - Does not launch Spark
# ============================================================

for required_name in [
    "SIX_CITY_AUDIT",
    "TRAINING_CITY_INPUTS",
    "EXPECTED_EPOCHS",
    "EXPECTED_CITY_COUNT",
    "EXPECTED_PARTITION_COUNT",
    "EXPECTED_ROWS_PER_PARTITION",
    "EXPECTED_TOTAL_ROWS",
    "LOCAL_ARTIFACT_ROOT",
]:
    if required_name not in globals():
        raise RuntimeError(
            required_name
            + " is unavailable. Run the corrected Cells 1-6 first."
        )


# ============================================================
# VALIDATE THE COMPLETED SIX-CITY AUDIT
# ============================================================

if (
    SIX_CITY_AUDIT.get(
        "partition_count"
    )
    != EXPECTED_PARTITION_COUNT
):
    raise RuntimeError(
        "Expected "
        + str(
            EXPECTED_PARTITION_COUNT
        )
        + " audited partitions, found "
        + str(
            SIX_CITY_AUDIT.get(
                "partition_count"
            )
        )
    )


if (
    SIX_CITY_AUDIT.get(
        "total_rows"
    )
    != EXPECTED_TOTAL_ROWS
):
    raise RuntimeError(
        "Expected "
        + str(
            EXPECTED_TOTAL_ROWS
        )
        + " audited rows, found "
        + str(
            SIX_CITY_AUDIT.get(
                "total_rows"
            )
        )
    )


partition_results = (
    SIX_CITY_AUDIT.get(
        "partitions",
        [],
    )
)


if (
    len(
        partition_results
    )
    != EXPECTED_PARTITION_COUNT
):
    raise RuntimeError(
        "Expected "
        + str(
            EXPECTED_PARTITION_COUNT
        )
        + " partition audit records, found "
        + str(
            len(
                partition_results
            )
        )
    )


if (
    len(
        TRAINING_CITY_INPUTS
    )
    != EXPECTED_CITY_COUNT
):
    raise RuntimeError(
        "Expected "
        + str(
            EXPECTED_CITY_COUNT
        )
        + " training cities, found "
        + str(
            len(
                TRAINING_CITY_INPUTS
            )
        )
    )


if SIX_CITY_AUDIT.get(
    "unexpected_layouts",
    [],
):
    raise RuntimeError(
        "Unexpected HDFS layouts were found: "
        + repr(
            SIX_CITY_AUDIT.get(
                "unexpected_layouts"
            )
        )
    )


# ============================================================
# RESOLVE THE EXACT SIX TRAINING-CITY SLUGS
# ============================================================

RESOLVED_CITY_SLUGS = [
    city_config[
        "city_slug"
    ]
    for city_config
    in TRAINING_CITY_INPUTS.values()
]


if (
    len(
        RESOLVED_CITY_SLUGS
    )
    != EXPECTED_CITY_COUNT
):
    raise RuntimeError(
        "Expected six resolved city slugs, found "
        + repr(
            RESOLVED_CITY_SLUGS
        )
    )


if (
    len(
        set(
            RESOLVED_CITY_SLUGS
        )
    )
    != EXPECTED_CITY_COUNT
):
    raise RuntimeError(
        "Resolved city slugs are not unique: "
        + repr(
            RESOLVED_CITY_SLUGS
        )
    )


audited_city_slugs = sorted(
    SIX_CITY_AUDIT.get(
        "audited_city_slugs",
        [],
    )
)


if (
    sorted(
        RESOLVED_CITY_SLUGS
    )
    != audited_city_slugs
):
    raise RuntimeError(
        "Resolved training-city slugs do not match "
        "the audited city slugs. Resolved="
        + repr(
            sorted(
                RESOLVED_CITY_SLUGS
            )
        )
        + ", audited="
        + repr(
            audited_city_slugs
        )
    )


# ============================================================
# BUILD COLUMN AND COVERAGE INVENTORIES
# ============================================================

city_columns = {}
city_coverage = {}


for partition_record in partition_results:
    city_slug = (
        partition_record[
            "city_slug"
        ]
    )

    city_columns.setdefault(
        city_slug,
        set(),
    ).update(
        partition_record[
            "columns"
        ]
    )

    city_coverage.setdefault(
        city_slug,
        set(),
    ).update(
        partition_record[
            "coverage_columns"
        ]
    )


if (
    len(
        city_columns
    )
    != EXPECTED_CITY_COUNT
):
    raise RuntimeError(
        "Expected column inventories for six cities, found "
        + repr(
            sorted(
                city_columns
            )
        )
    )


common_columns = sorted(
    set.intersection(
        *city_columns.values()
    )
)


all_columns = sorted(
    set.union(
        *city_columns.values()
    )
)


city_specific_columns = {
    city_slug: sorted(
        city_columns[
            city_slug
        ]
        - set(
            common_columns
        )
    )
    for city_slug
    in sorted(
        city_columns
    )
}


# ============================================================
# IDENTIFY SCHEMA VARIANTS AND REQUIRED HARMONIZATION
#
# Cell 6 found two schema variants:
#
# Variant 1:
#   Bengaluru, Hyderabad, Pune and Delhi NCR
#   - built_observed_grid_area_m2: decimal(27,1)
#   - built_coverage_fraction: decimal(27,6)
#
# Variant 2:
#   Chennai and Mumbai
#   - built_observed_grid_area_m2: double
#   - built_coverage_fraction: double
#
# Before unioning all six cities, both fields will be cast
# explicitly to double.
# ============================================================

schema_variant_count = (
    SIX_CITY_AUDIT.get(
        "schema_variant_count"
    )
)


if schema_variant_count != 2:
    raise RuntimeError(
        "Expected exactly two schema variants, found "
        + str(
            schema_variant_count
        )
    )


SCHEMA_HARMONIZATION_CASTS = {
    "built_observed_grid_area_m2": (
        "double"
    ),
    "built_coverage_fraction": (
        "double"
    ),
}


# ============================================================
# REPORT TO NOTEBOOK
# ============================================================

print(
    "Schema variants:",
    schema_variant_count,
)

print(
    "Common columns across all six cities:",
    len(
        common_columns
    ),
)

print(
    "All distinct columns across all six cities:",
    len(
        all_columns
    ),
)

print(
    "Coverage columns by city:"
)


for city_slug in sorted(
    city_coverage
):
    print(
        " -",
        city_slug,
        sorted(
            city_coverage[
                city_slug
            ]
        ),
    )


print(
    "Required schema harmonization casts:"
)


for (
    column_name,
    target_type,
) in SCHEMA_HARMONIZATION_CASTS.items():
    print(
        " -",
        column_name,
        "->",
        target_type,
    )


# ============================================================
# BUILD THE LOCAL INVENTORY REPORT
# ============================================================

training_city_records = {
    city_name: {
        "city_slug": (
            city_config[
                "city_slug"
            ]
        ),
        "layout": (
            city_config[
                "layout"
            ]
        ),
        "enriched_root": (
            city_config[
                "enriched_root"
            ]
        ),
    }
    for (
        city_name,
        city_config,
    ) in TRAINING_CITY_INPUTS.items()
}


report = {
    "training_city_count": (
        EXPECTED_CITY_COUNT
    ),
    "training_city_slugs": (
        RESOLVED_CITY_SLUGS
    ),
    "training_cities": (
        training_city_records
    ),
    "held_out_city": {
        "city_name": "Jaipur",
        "city_slug": "jaipur",
        "included_in_training": False,
        "purpose": (
            "held_out_generalization_test"
        ),
    },
    "expected_epochs": (
        EXPECTED_EPOCHS
    ),
    "expected_rows_per_partition": (
        EXPECTED_ROWS_PER_PARTITION
    ),
    "expected_partition_count": (
        EXPECTED_PARTITION_COUNT
    ),
    "expected_total_rows": (
        EXPECTED_TOTAL_ROWS
    ),
    "audit_summary": {
        "partition_count": (
            SIX_CITY_AUDIT[
                "partition_count"
            ]
        ),
        "total_rows": (
            SIX_CITY_AUDIT[
                "total_rows"
            ]
        ),
        "audited_city_slugs": (
            SIX_CITY_AUDIT[
                "audited_city_slugs"
            ]
        ),
        "bengaluru_partition_count": (
            SIX_CITY_AUDIT[
                "bengaluru_partition_count"
            ]
        ),
        "generalized_partition_count": (
            SIX_CITY_AUDIT[
                "generalized_partition_count"
            ]
        ),
        "unexpected_layouts": (
            SIX_CITY_AUDIT[
                "unexpected_layouts"
            ]
        ),
        "duplicate_grid_keys": 0,
        "null_grid_keys": 0,
    },
    "schema_compatibility": {
        "all_columns": (
            all_columns
        ),
        "common_columns": (
            common_columns
        ),
        "columns_by_city": {
            city_slug: sorted(
                columns
            )
            for (
                city_slug,
                columns,
            ) in city_columns.items()
        },
        "city_specific_columns": (
            city_specific_columns
        ),
        "coverage_columns_by_city": {
            city_slug: sorted(
                columns
            )
            for (
                city_slug,
                columns,
            ) in city_coverage.items()
        },
        "schema_variant_count": (
            schema_variant_count
        ),
        "schema_groups": (
            SIX_CITY_AUDIT.get(
                "schema_groups",
                [],
            )
        ),
        "harmonization_required": True,
        "harmonization_casts": (
            SCHEMA_HARMONIZATION_CASTS
        ),
    },
    "bengaluru_standardized_outputs_used": True,
    "bengaluru_legacy_outputs_used": False,
    "bengaluru_legacy_outputs_preserved": True,
    "created_at_utc": time.strftime(
        "%Y-%m-%dT%H:%M:%SZ",
        time.gmtime(),
    ),
    "hdfs_modified": False,
    "spark_job_launched_by_cell": False,
}


# ============================================================
# WRITE AND VERIFY THE LOCAL JSON INVENTORY
# ============================================================

LOCAL_ARTIFACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


inventory_path = (
    LOCAL_ARTIFACT_ROOT
    / "six_city_inventory.json"
)


inventory_path.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


if (
    not inventory_path.exists()
):
    raise RuntimeError(
        "The local inventory report was not created."
    )


if (
    inventory_path.stat().st_size
    <= 0
):
    raise RuntimeError(
        "The local inventory report is empty."
    )


try:
    frozen_inventory = json.loads(
        inventory_path.read_text(
            encoding="utf-8"
        )
    )

except json.JSONDecodeError as error:
    raise RuntimeError(
        "The local inventory report is not valid JSON."
    ) from error


if (
    frozen_inventory.get(
        "expected_total_rows"
    )
    != EXPECTED_TOTAL_ROWS
):
    raise RuntimeError(
        "The frozen inventory has an incorrect "
        "expected total-row count."
    )


if (
    len(
        frozen_inventory.get(
            "training_city_slugs",
            [],
        )
    )
    != EXPECTED_CITY_COUNT
):
    raise RuntimeError(
        "The frozen inventory has an incorrect "
        "training-city count."
    )


# ============================================================
# FINAL REPORT
# ============================================================

print(
    "Inventory report:",
    inventory_path,
)

print(
    "Inventory JSON validation: PASS"
)

print(
    "Cities verified: 6/6"
)

print(
    "Partitions verified: 60/60"
)

print(
    "Rows verified: 294000"
)

print(
    "Schema variants recorded: 2"
)

print(
    "Schema harmonization plan recorded: YES"
)

print(
    "Jaipur included in training: NO"
)

print(
    "Spark job launched by Cell 7: NO"
)

print(
    "HDFS modified: NO"
)

print(
    "DAY_1_SIX_CITY_INVENTORY_SUCCESS"
)

print(
    "CELL 7: SUCCESS"
)

Schema variants: 2
Common columns across all six cities: 48
All distinct columns across all six cities: 48
Coverage columns by city:
 - bengaluru ['built_coverage_fraction', 'population_coverage_fraction']
 - chennai_metro ['built_coverage_fraction', 'population_coverage_fraction']
 - delhi_ncr ['built_coverage_fraction', 'population_coverage_fraction']
 - hyderabad ['built_coverage_fraction', 'population_coverage_fraction']
 - mumbai_metro ['built_coverage_fraction', 'population_coverage_fraction']
 - pune_metro ['built_coverage_fraction', 'population_coverage_fraction']
Required schema harmonization casts:
 - built_observed_grid_area_m2 -> double
 - built_coverage_fraction -> double
Inventory report: /content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_inventory.json
Inventory JSON validation: PASS
Cities verified: 6/6
Partitions verified: 60/60
Rows verified: 294000
Schema variants recorded: 2
Schema harmonization plan recorded: YES
Jaipur included in training: NO
Spark job launc

Completed:

EC2/HDFS recovery

Six-city path resolution

All 60 partitions located

Every partition structurally validated

Every row and grid key validated

Cross-city schema compared

Harmonization plan frozen

Local inventory artifact created
